In [89]:
from youtube_transcript_api import YouTubeTranscriptApi
# from langchain_core.document_loaders
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [90]:
from langchain_openai import OpenAIEmbeddings,ChatOpenAI

In [91]:
# from langchain_ollama import OllamaLLM
# from langchain_ollama import OllamaEmbeddings

In [92]:
youtube_api=YouTubeTranscriptApi()
video_id='BOY65Sw8xL0'
video_transcripts=youtube_api.fetch(video_id)

In [93]:
print(type(video_transcripts))
# for i in video_transcripts:
#     print(i.text)

<class 'youtube_transcript_api._transcripts.FetchedTranscript'>


In [94]:
video_transcripts_text=' '.join(i.text for i in video_transcripts)
video_transcripts_text

"Hello guys, my name is Krishna and welcome to my YouTube channel. So guys today in this particular video we are going to discuss about the entire AI engineering road map for the upcoming years and this specific road map you know I do definitely make up every year you know and probably it's been time because currently you can see that there's lot of developments in the field of AI you know you have generative AI you have agentic AI now there are roles like AIFD so this kind of road map that I'm actually designing will target almost every roles right like AI engineer whether you are working as an LLM ops engineer or whether you go ahead and probably work as an AIFD engineer in the future right so all these things will be covered and I'll also be making sure that I provide you lot of resources uh free resources along with videos and many more things so please make sure that you watch this video till the end okay now let me quickly go ahead and share my screen and before I go ahead I real

In [95]:
splitter_obj=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)

In [96]:
transcripts_chunk=splitter_obj.split_text(video_transcripts_text)
transcripts_chunk

["Hello guys, my name is Krishna and welcome to my YouTube channel. So guys today in this particular video we are going to discuss about the entire AI engineering road map for the upcoming years and this specific road map you know I do definitely make up every year you know and probably it's been time because currently you can see that there's lot of developments in the field of AI you know you have generative AI you have agentic AI now there are roles like AIFD so this kind of road map that I'm",
 "road map that I'm actually designing will target almost every roles right like AI engineer whether you are working as an LLM ops engineer or whether you go ahead and probably work as an AIFD engineer in the future right so all these things will be covered and I'll also be making sure that I provide you lot of resources uh free resources along with videos and many more things so please make sure that you watch this video till the end okay now let me quickly go ahead and share my screen and",

In [97]:
transcripts_embedding=OpenAIEmbeddings(model='text-embedding-3-small')
# transcripts_ollama_embedding=OllamaEmbeddings(model='qwen3-embedding:0.6b')

In [98]:
print(type(transcripts_chunk))

<class 'list'>


In [99]:
yt_vectore_store=FAISS.from_texts(transcripts_chunk,transcripts_embedding)
# yt_ollama_vector_store=FAISS.from_texts(transcripts_chunk,transcripts_ollama_embedding)

In [100]:
yt_vectore_store.similarity_search(query='Tell me about video transcript about')
# yt_ollama_vector_store.similarity_search(query='Tell me about video transcript about')

[Document(id='f1c1018f-d7f6-4baa-adac-2cef17324d2c', metadata={}, page_content="about context engineering. So there you understand about prompt engineering, dynamic context, context comparation, system prompts, few short prompting, context selection, structured context, long-term context. So all these things I have explained in detail in all my playlist that I've created related to generative AI and agent. I also have one 11 hours course. Uh it it has gained millions of views just in 3 to four months. But don't worry, I will be probably showing you structured way how you"),
 Document(id='35ed8a37-8e1a-432c-9bad-54c3f96d75cf', metadata={}, page_content="and then you'll be able to talk. It will be able to take the context with respect to all the information that we have over here and it'll give able to give you the answer. So go ahead and check it out. Uh again with respect to free resources, everything is available over here, right? Any kind of course that you specifically want to check

In [101]:
yt_transcript_retriver=yt_vectore_store.as_retriever(search_type='mmr',search_kwargs={'k':3,'lambda_mult':0.5})
# yt_ollama_transcript_retriver=yt_ollama_vector_store.as_retriever(search_type='mmr',search_kwargs={'k':3,'lambda_mult':0.5})

In [123]:
query='Tell me about video transcript about'

In [103]:
results=yt_transcript_retriver.invoke(query)
# ollama_results=yt_ollama_transcript_retriver.invoke(query)
results
# ollama_results

[Document(id='f1c1018f-d7f6-4baa-adac-2cef17324d2c', metadata={}, page_content="about context engineering. So there you understand about prompt engineering, dynamic context, context comparation, system prompts, few short prompting, context selection, structured context, long-term context. So all these things I have explained in detail in all my playlist that I've created related to generative AI and agent. I also have one 11 hours course. Uh it it has gained millions of views just in 3 to four months. But don't worry, I will be probably showing you structured way how you"),
 Document(id='1ed15e9e-01a9-498b-bada-c0a7ee2d90e8', metadata={}, page_content="seeing task assess tool selection tool argument goal completion in rag eval you'll be talking about retrieval recall precision groundness faithfulness so these are some of the very important metrics in LM value you talk about accuracy relevance hallucination, toxicity and LLM as a judge, right? And based on the scenario, based on the app

In [104]:
for i in results:
    print(i.page_content)

# for i in ollama_results:
#     print(i.page_content)   

about context engineering. So there you understand about prompt engineering, dynamic context, context comparation, system prompts, few short prompting, context selection, structured context, long-term context. So all these things I have explained in detail in all my playlist that I've created related to generative AI and agent. I also have one 11 hours course. Uh it it has gained millions of views just in 3 to four months. But don't worry, I will be probably showing you structured way how you
seeing task assess tool selection tool argument goal completion in rag eval you'll be talking about retrieval recall precision groundness faithfulness so these are some of the very important metrics in LM value you talk about accuracy relevance hallucination, toxicity and LLM as a judge, right? And based on the scenario, based on the application that you have created, we create this entire evaluation pipeline. We'll be having some golden data sets, highly quality annotated data sets. We'll be
out 

In [105]:
prompt_template=PromptTemplate(template='''You are a helpful assistant that answers questions based on the context provided. If the context does not contain the answer, say "I do not know".\n\nContext: {context}\n\nQuestion: {question}\nAnswer:''',input_variables=['context','question'])

In [115]:
test_query='Tell me about context of video transcript'

In [116]:
required_context=' '.join(i.page_content for i in results)
# ollama_required_context=' '.join(i.page_content for i in ollama_results)

In [117]:
required_context
# ollama_required_context

"about context engineering. So there you understand about prompt engineering, dynamic context, context comparation, system prompts, few short prompting, context selection, structured context, long-term context. So all these things I have explained in detail in all my playlist that I've created related to generative AI and agent. I also have one 11 hours course. Uh it it has gained millions of views just in 3 to four months. But don't worry, I will be probably showing you structured way how you seeing task assess tool selection tool argument goal completion in rag eval you'll be talking about retrieval recall precision groundness faithfulness so these are some of the very important metrics in LM value you talk about accuracy relevance hallucination, toxicity and LLM as a judge, right? And based on the scenario, based on the application that you have created, we create this entire evaluation pipeline. We'll be having some golden data sets, highly quality annotated data sets. We'll be out

In [118]:
final_prompt=(prompt_template.invoke({'question': test_query,'context':required_context}))
# ollama_final_prompt=(prompt_template.invoke({'question': test_query,'context':ollama_required_context}))    

In [119]:
final_prompt
# ollama_final_prompt

StringPromptValue(text='You are a helpful assistant that answers questions based on the context provided. If the context does not contain the answer, say "I do not know".\n\nContext: about context engineering. So there you understand about prompt engineering, dynamic context, context comparation, system prompts, few short prompting, context selection, structured context, long-term context. So all these things I have explained in detail in all my playlist that I\'ve created related to generative AI and agent. I also have one 11 hours course. Uh it it has gained millions of views just in 3 to four months. But don\'t worry, I will be probably showing you structured way how you seeing task assess tool selection tool argument goal completion in rag eval you\'ll be talking about retrieval recall precision groundness faithfulness so these are some of the very important metrics in LM value you talk about accuracy relevance hallucination, toxicity and LLM as a judge, right? And based on the sce

In [120]:
chat_model=ChatOpenAI(model='gpt-3.5-turbo',temperature=0.2)
# ollama_chat_model=OllamaLLM(model='llama3.2:3b',temperature=0.2)

In [121]:
final_answer=chat_model.invoke(final_prompt)
# ollama_final_answer=ollama_chat_model.invoke(ollama_final_prompt)

In [122]:
print(final_answer)
print(final_answer.content)

# print(ollama_final_answer)
# # print(ollama_final_answer.content)

content='The context of the video transcript is related to context engineering in the field of generative AI and agent development. It discusses various aspects such as prompt engineering, dynamic context, context comparison, system prompts, context selection, structured context, long-term context, and evaluation metrics like retrieval recall precision, groundness, faithfulness, accuracy, relevance, hallucination, toxicity, and LLM. The speaker also mentions creating an evaluation pipeline based on the scenario and application, using high-quality annotated datasets and AI forward deployed engineer boot camp.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 282, 'total_tokens': 387, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_to